# Tutorial for Analog Forecasting Project

#### This notebook will step through an overview of the analog method, what is novel about our approach, and how to run a simple experiment!

## Analog Overview

#### The idea behind analog forecasting is that similar climate states should evolve similarly, at least for some amount of time

#### For example, if I want to forecast the temperature in Fort Collins tomorrow, I could look at the entire history of Fort Collins and pick a time in the past that looks the most similar to today -- the day after that most similar match is my prediction for tomorrow

#### In our case, we are looking ahead years or even decades instead of days, but the idea is the same

## Our Approach

#### Analog forecasting went out of fashion fairly quickly for a few reasons:
- ##### There weren't enough potential analogs to find a really good match,
- ##### And it was unclear how to define "match" (should it match everywhere in the globe? just in Fort Collins?)

#### To get around these issues, our method
- ##### Uses Earth system models, not just actual observations, for the library of analogs (we have a lot of these ESMs),
- ##### And uses machine learning to determine what areas are most important to match on

#### Below is a schematic showing the machine learning part -- this is fairly complicated, so don't expect it to make sense right away

<div align="center">
    <img src="figures/main_schematic.png" width=60% height=60%>
</div>

#### 1. Two maps are taken (not necessarily from the same model, member, or time) and their difference is scaled by a randomly initialized mask of weights. This effectively highlights some areas as important for matching, and minimizes other areas as less important for matching when determining what is a good analog.
#### 2. The mean squared value of this weighted difference map is passed to a neural network tasked with predicting the mean-squared-error between the two maps at a future time
#### 3. The randomly initialized mask of weights is updated through training until the network predicts future mse from weighted map difference well
#### 4. The final mask of weights can then be used in an analog forecasting framework

## now, onto an example

In [9]:
import os
os.environ['ESMFMKFILE'] = '/Users/jlandsbe/anaconda3/envs/analog-env/lib/esmf.mk'

In [10]:
# general imports
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import tensorflow as tf
import random
# mask-analog imports
import run_experiments as rexp
import build_model
import train_model
import make_mask as mask
import best_analogs as ba
import cftime

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

%matplotlib inline

# workflow for a new experiment

### 1. create a dictionary entry for the new experiment

In [11]:
# first, create an Experiments object which allows you to define a new experiment and run it
# the json file it is referencing saves all of your experiments in a file, so they can be run, re-run, referenced, etc.
exp_object = rexp.Experiments('experiments.json')

# if you want to see all the models, run exp_object.list_variables
# if you want to see all the named regions, run exp_object.list_regions

# now, define a new experiment with all the settings you may want to change
# the full list of settings can be found by running help(exp_object.make_exp_dictionary) in a cell
# the main settings you will want to change are shown in the example
already_made = 0
if already_made == 0:
    new_exp = exp_object.make_exp_dictionary('example_exp', leadtime=5, window=1, variable_list=['tas',], analog_models=['CanESM5', 'MIROC-ES2L'], soi_models=['CanESM5', 'MIROC-ES2L'], split=[5,2,2,1], prediction_nodes=[5,5,5], prediction_activation='elu', learning_rate=0.001, output_period='1Y', region='western us', baseline=[1961, 1990])
# this adds the experiment to the json file
exp_object.add_experiment(new_exp)
already_made = 1

AssertionError: experiment with that name already exists

### 2. preprocesses the data

i've actually run the preprocessing already for this example, so there is already a saved file it will load

once you have access to the full dataset, the preprocessing will actually go through all the steps

In [ ]:
# NOTE: you may need to run this in a terminal (we can discuss) when the amount of data gets larger -- for this example, it will be fine to run in the notebook

# first, preprocess the data according to the settings in the new experiment dictionary
(analog_inputs, analog_target, training_inputs, training_target,
validation_inputs, validation_target, testing_inputs, testing_target, lat, lon) = exp_object.get_experiment_data('example_exp', overwrite_data=False)
# reshape for multiple variables, since tensorflow (the machine learning code) wants it this way
analog_inputs = np.transpose(analog_inputs, (0, 3, 1, 2))
training_inputs = np.transpose(training_inputs, (0, 3, 1, 2))
validation_inputs = np.transpose(validation_inputs, (0, 3, 1, 2))
testing_inputs = np.transpose(testing_inputs, (0, 3, 1, 2))

loading variables: tas
from models: CanESM5 MIROC-ES2L
directory: /Users/jlandsbe/Downloads/Martin_analogs/mask-analog-projections/raw_data/monthly/CanESM5/tas does not exist
continuing without this dataset
directory: /Users/jlandsbe/Downloads/Martin_analogs/mask-analog-projections/raw_data/monthly/MIROC-ES2L/tas does not exist
continuing without this dataset
#################################################################
processing analog input dataset
#################################################################
down-sampling dataset from 1M to 1Y


UnboundLocalError: cannot access local variable 'model' where it is not associated with a value

we will play around with different `leadtime`, `window`, `analog_models`/`soi_models`, and `region`, so you'll get to know these well, but there are many

potential preprocessing steps that determine the goal of the predictions and how well they work -- these are all listed in the `_data_processing.py` file

to start out we will stick to the basic processing steps, but as the project moves along we can discuss including different steps

### 3. build the neural network model

In [ ]:
# get the settings for building the model
experiment_settings = exp_object.get_experiment('example_exp')

## build the model
# ensure model names are consistent by setting a random seed
tf.keras.backend.clear_session()
np.random.seed(experiment_settings["seed"])
random.seed(experiment_settings["seed"])
tf.random.set_seed(experiment_settings["seed"])
# build the full model for learning the mask: mask, dissimilarity, prediction models
model, _, _, _ = build_model.build_interp_model(experiment_settings, [training_inputs, analog_inputs])

: 

### 4. train the model using the processed data

In [ ]:
## train the model using the processed data
model, fit_summary, history = train_model.train_model(
    experiment_settings,
    model,
    analog_inputs,
    analog_target,
    training_inputs,
    training_target,
    validation_inputs,
    validation_target,
    overwrite_model=True
    )

: 

### 5. save the mask

In [ ]:
## get and save the mask
mask = mask.make_mask('example_exp', model, exp_object.processed_data, overwrite_mask=True)

: 

In [ ]:
# you can get a sneak peek at the mask by loading in the saved mask and plotting it

# load the mask file
mask = xr.load_dataset('saved_masks/example_exp_mask.nc')
# set up the plot size, projection
fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=215.))
data_crs = ccrs.PlateCarree()
# this gets the continents
land_feature = cfeature.NaturalEarthFeature(
    category='physical',
    name='land',
    scale='110m',
    facecolor='none',
    edgecolor='k',
    linewidth=1,
)
ax.add_feature(land_feature)
# now plot the mask weights on top of the map
image = ax.pcolormesh(mask.lon, mask.lat, mask.tas[0], transform=data_crs, cmap='Greys')
plt.show()

: 

this can also all be done with the make_mask function -- `exp_object.make_mask('example_exp')` -- which runs through most of the code above!

it takes the raw data, processes it according to the settings in the experiment dictionary, builds the machine learning model, trains the model using the

data, then saves the mask (note the trained model and processed data are also saved)

### 6. generate a library of analogs and SOIs (states of interest)

In [ ]:
# now that the mask has been calculated, we can make regular analog forecasts using the mask

# this takes the raw data specified in model_list (and n_members, with -1 meaning grab all available members)
# and processes it using most of the same steps that were used for the mask training data (some processing steps are not necessary and are autmoatically removed)
# the processed data is saved in the saved_analogs folder
library = ba.BestAnalogs('example_exp', model_list=['CanESM5', 'MIROC-ES2L'], n_members=10)
# this is primarily the analog library, but for convenience, the SOIs can be added here as well (since they need to be processed the same anyway)
# here, I am adding the BEST (Berkeley Earth Surface Temperatures) observational data set to use as an SOI (what we are predicting)
library = ba.BestAnalogs('example_exp', model_list=["BEST"], n_members=-1, data_period='yearly')

: 

### 7. calculate the best analogs for a given SOI

In [ ]:
# finally, this functions ranks all the potential analogs for the BEST data (all times) and saves those for later use
# this step takes a while and can use a lot of memory, so this will need to be run in terminal when the data gets larger (fine to run here for now)
library.get_all_soi_analogs("BEST", 0)

# this is saved in the same file as the library, so it is easier to reference the best analogs directly

: 

### 8. make various plots

i've coded up several figures, but feel free (even encouraged) to make your own plots!

In [ ]:
# first, load up the library -- not passing any models means it will just load the previously processed data
# (if you run with the same models it will duplicate them -- I will fix this at some point, for now avoid doing that)
library = ba.BestAnalogs('example_exp')

# here is a prediction plot (which SOI to plot, which member, the variable, and number of analogs to show the mean of)
library.plot_projection("BEST", 0, 'tas', 2)

: 

note: this probably doesn't look like that good of a prediction, but that is because i've kept the training dataset and analog library very small so it will run quickly

In [ ]:
# here is the distribution of how often each model was picked as a top 10 analog
library.plot_model_distribution("BEST", 0, 'tas', 10)

: 

In [ ]:
# or plot how often each model is picked as a function of time
library.plot_model_times("BEST", 0, 'tas', 10)

: 

In [ ]:
# or plot the mask again, to compare to the previous
library.plot_mask(colorbar=False)

: 

: 